# Growth rate, dominant wavelength and eddy lifetime from the MOM6 simulations

Computes, for every experiment (Section 2.3 of the manuscript):
- the domain-integrated EKE $E(t)$ in a cross-shelf band that follows the jet centre,
- the reference time $t_0$ (`idx0`): first local minimum of $E$ followed by sustained growth,
- the growth rate $\ln[E(t)/E(t_0)] / 2(t-t_0)$, smoothed with a Gaussian filter ($\sigma$ = 24 h),
- the dominant wavelength at every output time, from the y-summed along-shelf power spectrum of $u'$,
- the eddy lifetime end: the last time the wavelength at peak growth stays within 10% of its value.

Results are saved to a NetCDF file, which is read by `05_fig2...` and `07_figs4-6...`.
Experiments are processed in concurrent batches on a Dask cluster.

## 1. Imports & Dask cluster

In [1]:
import intake
import xarray as xr

import xgcm
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.lines import Line2D
from pathlib import Path
import gc
import warnings
import dask
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from dask.distributed import Client
from scipy.ndimage import gaussian_filter1d

warnings.filterwarnings('ignore')

OUT_DIR = Path('idealize_nondim_plots')
OUT_DIR.mkdir(parents=True, exist_ok=True)

DX_M             = 300   # m per grid cell
CATALOG_TEMPLATE = '/home/130/ae7501/intake_style_datastore/{test}.json'

In [2]:
# Dask LocalCluster
client = Client()
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 7
Total threads: 28,Total memory: 251.19 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:42467,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:35801,Total threads: 4
Dashboard: /proxy/37045/status,Memory: 35.88 GiB
Nanny: tcp://127.0.0.1:45713,


2026-10-18 00:03:14,074 - distributed.nanny - WARNING - Restarting worker
2026-10-18 00:03:14,181 - distributed.nanny - WARNING - Restarting worker
2026-10-18 00:03:14,192 - distributed.nanny - WARNING - Restarting worker
2026-10-18 00:03:14,221 - distributed.nanny - WARNING - Restarting worker
2026-10-18 00:03:14,345 - distributed.nanny - WARNING - Restarting worker
2026-10-18 00:03:14,362 - distributed.nanny - WARNING - Restarting worker
2026-10-18 00:03:14,666 - distributed.nanny - WARNING - Restarting worker
2026-10-18 00:08:22,588 - distributed.nanny - WARNING - Restarting worker
2026-10-18 00:08:22,603 - distributed.nanny - WARNING - Restarting worker
2026-10-18 00:08:22,649 - distributed.nanny - WARNING - Restarting worker
2026-10-18 00:08:22,690 - distributed.nanny - WARNING - Restarting worker
2026-10-18 00:08:22,731 - distributed.nanny - WARNING - Restarting worker
2026-10-18 00:08:22,849 - distributed.nanny - WARNING - Restarting worker
2026-10-18 00:08:22,870 - distributed.

## 2. Grid setup & analysis parameters

In [3]:
static_path = Path('/g/data/nm03/ae7501/mom6_outputs/from_scratch/idealized/test170-7603273f/output000/ocean_static.nc')
o_static = xr.open_dataset(static_path).drop_vars(['time'], errors='ignore')
o_static.coords['xh'].attrs.update(axis='X')
o_static.coords['xq'].attrs.update(axis='X', c_grid_axis_shift=0.5)
o_static.coords['yh'].attrs.update(axis='Y')
o_static.coords['yq'].attrs.update(axis='Y', c_grid_axis_shift=0.5)
grid = xgcm.Grid(o_static)

In [4]:
dx             = 0.3   # grid-cell units (NOT metres); DX_M handles metres
F_CORIOLIS     = -1e-5  # s^-1
yh_slice       = slice(170000, 230000)   # window for a jet centred at YJ_REF_M
wavelength_min = 2
wavelength_max = 400
time_start     = 0
time_end       = 4350
time_step      = 1
DT_OUTPUT_S     = 3600.0   # s per time_idx step (hourly output)
time_indices   = range(time_start, time_end, time_step)
hovmoller_time_end = 1000

# Jet-following yh window: yh_slice (170-230 km) is defined for a jet centred at
# YJ_REF_M = 200 km. When the jet centre moves (test263-274) the window moves with it,
# keeping the same offsets from the jet core and the same number of yh points.
IC_ROOT          = Path('/g/data/nm03/ae7501/mom6_outputs/from_scratch/idealized')
YJ_REF_M         = 200000.0    # jet centre (m) that yh_slice is defined around
TRACK_JET_CENTRE = True        # False -> same yh_slice for every test

def jet_centre_y(test):
    """y of the jet core in MOM_IC.nc: location of peak shear |du/dy| of the top-layer zonal-mean u
    (u is a tanh step from -U to +U, so the jet sits where u changes fastest, not where |u| is largest).
    Refined with a |du/dy|-weighted centroid over points >= 50% of the peak."""
    matches = sorted(IC_ROOT.glob(f'{test}-*'))
    if not matches:
        raise FileNotFoundError(f'No archive dir for {test} under {IC_ROOT}')
    with xr.open_dataset(matches[0] / 'output000' / 'MOM_IC.nc') as ds:
        u = ds['u']
        if 'Time' in u.dims:
            u = u.isel(Time=0)
        z = next(d for d in u.dims if d in ('zl', 'Layer', 'z_l'))
        y = next(d for d in u.dims if d in ('yh', 'lath', 'lat', 'y'))
        x = next(d for d in u.dims if d in ('xq', 'xh', 'lonq', 'lon', 'x'))
        u_prof = u.isel({z: 0}).mean(dim=x).values
        yv     = u[y].values.astype(float)
    shear = np.abs(np.gradient(u_prof, yv))
    k     = int(shear.argmax())
    lo, hi = k, k
    while lo > 0 and shear[lo - 1] >= 0.5 * shear[k]:
        lo -= 1
    while hi < len(shear) - 1 and shear[hi + 1] >= 0.5 * shear[k]:
        hi += 1
    w = shear[lo:hi + 1]
    return float((yv[lo:hi + 1] * w).sum() / w.sum())

_yh     = o_static['yh'].values
_in_ref = (_yh >= yh_slice.start) & (_yh <= yh_slice.stop)
_n_win  = int(_in_ref.sum())
_YJ_REF = YJ_REF_M
_di     = int(np.argmax(_in_ref)) - int(np.abs(_yh - _YJ_REF).argmin())  # window start rel. to jet-core index

print(f'Reference jet centre: {_YJ_REF/1e3:.2f} km | '
      f'window {yh_slice.start/1e3:.0f}-{yh_slice.stop/1e3:.0f} km = '
      f'{(yh_slice.start-_YJ_REF)/1e3:+.1f} to {(yh_slice.stop-_YJ_REF)/1e3:+.1f} km from core, {_n_win} pts')

def yh_slice_for(test):
    if not TRACK_JET_CENTRE:
        return yh_slice
    yc = jet_centre_y(test)
    i0 = int(np.clip(int(np.abs(_yh - yc).argmin()) + _di, 0, len(_yh) - _n_win))
    return slice(float(_yh[i0]), float(_yh[i0 + _n_win - 1]))

Reference jet centre: 200.00 km | window 170-230 km = -30.0 to +30.0 km from core, 200 pts


## 3. Per-experiment compute — one at a time, delete immediately



In [5]:
def smooth_gaussian(x, sigma=24):
    filled = np.where(np.isfinite(x), x, np.nanmean(x))
    return gaussian_filter1d(filled.astype(float), sigma=sigma)

all_dominant_wavelengths = {}
all_growth_rates         = {}
all_initial_time_idx     = {}   # reference time t0
all_lifetime_end_time_idx = {}  # end of eddy lifetime
all_growth_peak_hit_wall = {}
all_lifetime_hit_wall    = {}

# Number of experiments processed concurrently
BATCH_SIZE = 6

# Experiments test160-274; any experiment without an intake catalog JSON is skipped

all_tests        = []
tests_no_catalog = []
for i in list(range(160,275)):
    test = f'test{i}'
    if Path(CATALOG_TEMPLATE.format(test=test)).exists():
        all_tests.append(test)
    else:
        tests_no_catalog.append(test)
if tests_no_catalog:
    print(f'Catalog not found (skipping): {", ".join(tests_no_catalog)}')
print(f'Experiments to process: {len(all_tests)}')

# Jet-following yh window per experiment
YH_SLICES = {}
for t in all_tests:
    YH_SLICES[t] = yh_slice_for(t)
    print(f'  {t}: yh window {YH_SLICES[t].start/1e3:7.2f} - {YH_SLICES[t].stop/1e3:7.2f} km')

# FFT wavelength arrays, shared across threads (set on the first experiment)
_fft_cache_lock = threading.Lock()
_n_xh  = None
_wl    = None
_valid = None

# Lock for writing results
_results_lock = threading.Lock()

def process_one(test):
    """Process a single experiment: load u/v, compute EKE, dominant wavelength,
    growth rate, reference time and lifetime end."""
    global _n_xh, _wl, _valid

    catalog_path = CATALOG_TEMPLATE.format(test=test)
    esm_ds = intake.open_esm_datastore(
        catalog_path,
        columns_with_iterables=[
            'variable', 'variable_long_name',
            'variable_standard_name',
            'variable_cell_methods', 'variable_units',
        ]
    )

    u_ds = esm_ds.search(variable='u').to_dask(chunks={'time': 50})
    v_ds = esm_ds.search(variable='v').to_dask(chunks={'time': 50})

    if 'u' not in u_ds or 'v' not in v_ds:
        u_ds.close(); v_ds.close()
        raise RuntimeError(f'Missing u or v in {test}')

    u     = u_ds['u']
    v     = v_ds['v']

    # grid.interp needs a single chunk along the interpolated axis (X for u, Y for v)
    u_x_dims = [d for d in u.dims if d in ('xh', 'xq')]
    v_y_dims = [d for d in v.dims if d in ('yh', 'yq')]
    u = u.chunk({d: -1 for d in u_x_dims})
    v = v.chunk({d: -1 for d in v_y_dims})

    u_int = grid.interp(u, 'X', boundary='extend')
    v_int = grid.interp(v, 'Y', boundary='extend')

    # Wavelength: FFT u'(y,x) along x at each timestep -> power |FFT|^2 -> sum over y
    # -> dominant wavelength. u and v are summed over depth and restricted to the
    # jet-following yh window, used for both the wavelength and the EKE.
    u_sum     = u_int.sum(dim='zl')
    v_sum     = v_int.sum(dim='zl')
    u_reduced = u_sum.sel(yh=YH_SLICES[test])   # jet-following window
    v_reduced = v_sum.sel(yh=YH_SLICES[test])

    u_prime_reduced = u_reduced - u_reduced.mean(dim='xh')   # for the wavelength
    v_prime_reduced = v_reduced - v_reduced.mean(dim='xh')   # for the EKE

    eke_lazy = (0.5 * (u_prime_reduced ** 2 + v_prime_reduced ** 2)).mean(dim='xh').sum(dim='yh')

    # Only the timesteps used by the FFT are loaded for u'; EKE uses the full series
    max_t        = u.sizes['time']
    loop_indices = [t for t in time_indices if t < max_t]

    u_prime_for_fft = u_prime_reduced.isel(time=loop_indices)

    # Both computed in one pass
    u_prime_da, eke_ds = dask.compute(u_prime_for_fft, eke_lazy)

    # Vectorised FFT; cache the wavelength arrays
    u_prime_np = u_prime_da.values   # (n_loop, yh, xh)
    n_xh       = u_prime_np.shape[-1]
    with _fft_cache_lock:
        if n_xh != _n_xh:
            _n_xh  = n_xh
            freq   = np.fft.rfftfreq(n_xh, d=dx)
            with np.errstate(divide='ignore', invalid='ignore'):
                _wl = np.where(freq != 0, 1.0 / freq, np.inf)
            _valid = (_wl >= wavelength_min) & (_wl <= wavelength_max)
        wl, valid = _wl.copy(), _valid.copy()

    fft_all      = np.fft.rfft(u_prime_np, axis=-1)   # (n_loop, yh, n_freq)
    power_all    = np.abs(fft_all) ** 2                # power spectrum of u'
    power_summed = power_all.sum(axis=1)               # summed over y
    power_masked = np.where(valid[np.newaxis, :], power_summed, -1.0)
    peak_indices = np.argmax(power_masked, axis=1)
    dom_wl       = wl[peak_indices]

    da_wl = xr.DataArray(
        data   = dom_wl,
        coords = {'time_idx': np.array(loop_indices)},
        dims   = ['time_idx'],
        name   = 'dominant_wavelength',
        attrs  = {'units': 'km', 'long_name': 'Dominant Wavelength'},
    )

    # EKE growth rate, referenced to the first local minimum of EKE followed by sustained growth
    E      = eke_ds.values.astype(np.float64)
    t_raw  = eke_ds.time.values
    dt_sec = (t_raw - t_raw[0]).astype('timedelta64[s]').astype(np.float64)

    # idx0: first local minimum where (a) E trends upward over check_window steps and
    # (b) E does not drop below the candidate value within sustain_window steps
    check_window   = 10
    sustain_window = 50
    sustain_tol    = 0.0

    local_min_idx = np.where((E[1:-1] < E[:-2]) & (E[1:-1] < E[2:]))[0] + 1

    idx0 = None
    rejected_false_starts = []
    for cand in local_min_idx:
        end_idx = cand + check_window
        if end_idx >= len(E):
            break

        window = E[cand:end_idx + 1]
        slope  = np.polyfit(np.arange(len(window)), window, 1)[0]
        if slope <= 0:
            continue

        sustain_end = min(cand + sustain_window, len(E))
        if np.any(E[cand:sustain_end] < E[cand] - sustain_tol):
            rejected_false_starts.append(cand)
            continue  # false start

        idx0 = cand
        break

    if idx0 is None:
        # Fallback: global minimum after t=0
        idx0 = 1 + np.argmin(E[1:])
        print(f'  {test}: no local minimum passed both checks -- falling back to global-min idx0={idx0}', flush=True)
    elif rejected_false_starts:
        print(f'  {test}: rejected {len(rejected_false_starts)} false-start candidate(s) at '
              f'{rejected_false_starts} before accepting idx0={idx0}', flush=True)

    E0 = E[idx0]
    t0 = dt_sec[idx0]

    growth_grad_raw = np.full_like(E, np.nan, dtype=float)
    with np.errstate(divide='ignore', invalid='ignore'):
        dt_from_ref = dt_sec - t0
        valid = (dt_from_ref > 0) & (E > 0)
        growth_grad_raw[valid] = np.log(E[valid] / E0) / (2.0 * dt_from_ref[valid])

    growth_cum = smooth_gaussian(growth_grad_raw, sigma=24)

    # Lifetime: last timestep (scanning forward from the peak growth rate) at which the
    # dominant wavelength is within tol_frac of its value at peak growth; lifetime = that - idx0.
    # 1. peak_tidx: timestep of max growth rate, searched from idx0 to t_end=4000
    PEAK_SEARCH_TSTART = 0
    PEAK_SEARCH_TEND   = 4000
    search_lo = max(PEAK_SEARCH_TSTART, idx0)
    search_hi = min(PEAK_SEARCH_TEND, len(growth_cum))
    if search_hi > search_lo:
        peak_tidx = search_lo + int(np.nanargmax(growth_cum[search_lo:search_hi]))
    else:
        peak_tidx = idx0

    # 2. lambda_peak: dominant wavelength nearest to peak_tidx
    loop_indices_arr = np.array(loop_indices)
    nearest_pos = int(np.argmin(np.abs(loop_indices_arr - peak_tidx)))
    lambda_peak = dom_wl[nearest_pos]

    # 3. Scan forward, tolerating up to max_gap consecutive misses
    tol_frac = 0.1
    max_gap  = 3
    tol      = tol_frac * lambda_peak
    gap      = 0
    idx_lifetime_end = int(loop_indices_arr[nearest_pos])
    for pos in range(nearest_pos, len(dom_wl)):
        if abs(dom_wl[pos] - lambda_peak) <= tol:
            idx_lifetime_end = int(loop_indices_arr[pos])
            gap = 0
        else:
            gap += 1
            if gap > max_gap:
                break

    # Data-completeness flags: did the peak or the lifetime end hit the end of the record?
    WALL_TOL = 3

    growth_peak_boundary  = min(PEAK_SEARCH_TEND, len(growth_cum) - 1)
    growth_peak_hit_wall  = bool(peak_tidx >= growth_peak_boundary - WALL_TOL)

    wl_record_end        = int(loop_indices_arr[-1])
    lifetime_hit_wall    = bool(idx_lifetime_end >= wl_record_end - WALL_TOL)

    if growth_peak_hit_wall:
        print(f'  [WALL] {test}: growth-rate peak (t_idx={peak_tidx}) hit the search '
              f'window/record boundary ({growth_peak_boundary}) -- may not be a genuine peak.', flush=True)
    if lifetime_hit_wall:
        print(f'  [WALL] {test}: lifetime end (t_idx={idx_lifetime_end}) hit the wavelength '
              f'record boundary ({wl_record_end}) -- lifetime may be an underestimate.', flush=True)

    da_gr = xr.DataArray(
        data   = growth_cum,
        coords = {'time_idx': np.arange(len(E))},
        dims   = ['time_idx'],
        name   = 'growth_rate',
        attrs  = {'units': 's-1', 'long_name': 'gradient EKE Growth Rate'},
    )

    u_ds.close(); v_ds.close()
    del u, v, u_sum, v_sum, u_reduced, v_reduced, u_prime_reduced, v_prime_reduced
    del u_prime_for_fft, u_prime_da, u_prime_np
    del eke_lazy, eke_ds, E, t_raw, dt_sec, growth_cum
    gc.collect()

    return test, da_wl, da_gr, idx0, idx_lifetime_end, growth_peak_hit_wall, lifetime_hit_wall

# Parallel batch loop: experiments in a batch run concurrently; workers are restarted between batches

batches    = [all_tests[i:i+BATCH_SIZE] for i in range(0, len(all_tests), BATCH_SIZE)]
n_done     = 0
n_failed   = 0

for batch_num, batch in enumerate(batches):
    print(f'\n[Batch {batch_num+1}/{len(batches)}] {batch}', flush=True)

    with ThreadPoolExecutor(max_workers=BATCH_SIZE) as pool:
        futures = {pool.submit(process_one, test): test for test in batch}

        for future in as_completed(futures):
            test = futures[future]
            try:
                (test_out, da_wl, da_gr, idx0_out, idx_lifetime_end_out,
                 growth_peak_hit_wall_out, lifetime_hit_wall_out) = future.result()
                with _results_lock:
                    all_dominant_wavelengths[test_out]   = da_wl
                    all_growth_rates[test_out]           = da_gr
                    all_initial_time_idx[test_out]       = int(idx0_out)
                    all_lifetime_end_time_idx[test_out]  = int(idx_lifetime_end_out)
                    all_growth_peak_hit_wall[test_out]   = bool(growth_peak_hit_wall_out)
                    all_lifetime_hit_wall[test_out]      = bool(lifetime_hit_wall_out)
                n_done += 1
                print(f'  [OK] {test_out}  ({n_done}/{len(all_tests)})', flush=True)
            except Exception as e:
                n_failed += 1
                print(f'  [FAIL] {test}: {e}', flush=True)

    print(f'  [restart] Clearing worker caches after batch {batch_num+1}...', flush=True)
    client.restart()

print(f'\nDone. {len(all_growth_rates)} succeeded, {n_failed} failed.')

n_growth_wall = sum(all_growth_peak_hit_wall.values())
n_lifetime_wall = sum(all_lifetime_hit_wall.values())
print(f'\n[FLAG SUMMARY]')
print(f'  Growth-rate peak hit the boundary (not a genuine peak): {n_growth_wall}/{len(all_growth_rates)} experiments')
print(f'  Lifetime end hit the record boundary (underestimate):    {n_lifetime_wall}/{len(all_growth_rates)} experiments')
if n_growth_wall:
    print(f'    -> growth: {sorted(t for t, v in all_growth_peak_hit_wall.items() if v)}')
if n_lifetime_wall:
    print(f'    -> lifetime: {sorted(t for t, v in all_lifetime_hit_wall.items() if v)}')

Catalog not found (skipping): test241, test262
Experiments to process: 113
  test160: yh window  170.25 -  229.95 km
  test161: yh window  170.25 -  229.95 km
  test162: yh window  170.25 -  229.95 km
  test163: yh window  170.25 -  229.95 km
  test164: yh window  170.25 -  229.95 km
  test165: yh window  170.25 -  229.95 km
  test166: yh window  170.25 -  229.95 km
  test167: yh window  170.25 -  229.95 km
  test168: yh window  170.25 -  229.95 km
  test169: yh window  170.25 -  229.95 km
  test170: yh window  170.25 -  229.95 km
  test171: yh window  170.25 -  229.95 km
  test172: yh window  170.25 -  229.95 km
  test173: yh window  170.25 -  229.95 km
  test174: yh window  170.25 -  229.95 km
  test175: yh window  170.25 -  229.95 km
  test176: yh window  170.25 -  229.95 km
  test177: yh window  170.25 -  229.95 km
  test178: yh window  170.25 -  229.95 km
  test179: yh window  170.25 -  229.95 km
  test180: yh window  170.25 -  229.95 km
  test181: yh window  170.25 -  229.95 km
 

In [6]:
# Save the results

SAVE_DIR = Path('/g/data/nm03/ae7501/mom6_outputs/processed_results')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Concatenate along 'experiment'; different time_idx lengths are padded with NaN
gr_list  = [da.expand_dims(experiment=[test]) for test, da in all_growth_rates.items()]
wl_list  = [da.expand_dims(experiment=[test]) for test, da in all_dominant_wavelengths.items()]

growth_rate_da         = xr.concat(gr_list, dim='experiment')
dominant_wavelength_da = xr.concat(wl_list, dim='experiment')

initial_time_idx_da = xr.DataArray(
    data   = [all_initial_time_idx[test] for test in all_growth_rates.keys()],
    coords = {'experiment': list(all_growth_rates.keys())},
    dims   = ['experiment'],
    name   = 'initial_time_idx',
    attrs  = {'long_name': 'time_idx of the reference (initial) energy point used for growth-rate calc'},
)

# Eddy lifetime = lifetime_end_time_idx - initial_time_idx
lifetime_end_time_idx_da = xr.DataArray(
    data   = [all_lifetime_end_time_idx[test] for test in all_growth_rates.keys()],
    coords = {'experiment': list(all_growth_rates.keys())},
    dims   = ['experiment'],
    name   = 'lifetime_end_time_idx',
    attrs  = {'long_name': 'last time_idx at which the max-growth-rate dominant wavelength is still dominant'},
)

# Data-completeness flags, saved as int8 (0/1)
growth_peak_hit_wall_da = xr.DataArray(
    data   = [int(all_growth_peak_hit_wall[test]) for test in all_growth_rates.keys()],
    coords = {'experiment': list(all_growth_rates.keys())},
    dims   = ['experiment'],
    name   = 'growth_peak_hit_wall',
    attrs  = {'long_name': 'True (1) if the growth-rate peak hit the search window/record boundary'},
).astype('int8')

lifetime_hit_wall_da = xr.DataArray(
    data   = [int(all_lifetime_hit_wall[test]) for test in all_growth_rates.keys()],
    coords = {'experiment': list(all_growth_rates.keys())},
    dims   = ['experiment'],
    name   = 'lifetime_hit_wall',
    attrs  = {'long_name': 'True (1) if lifetime end hit the wavelength record boundary'},
).astype('int8')

ds_out = xr.Dataset({
    'growth_rate':             growth_rate_da,
    'dominant_wavelength':     dominant_wavelength_da,
    'initial_time_idx':        initial_time_idx_da,
    'lifetime_end_time_idx':   lifetime_end_time_idx_da,
    'growth_peak_hit_wall':    growth_peak_hit_wall_da,
    'lifetime_hit_wall':       lifetime_hit_wall_da,
})

out_path = SAVE_DIR / '300_nondimplot_data_13_10.nc'
ds_out.to_netcdf(out_path)

print(f'Saved {len(all_growth_rates)} experiments to {out_path}')

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.
Saved 113 experiments to /g/data/nm03/ae7501/mom6_outputs/processed_results/300_nondimplot_data_13_10.nc
